In [2]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib

# Load input data and trained model

In [7]:
def compute_a(y):
    return y.sum() / len(y)

def compute_ahat(yhat, y_vals, yhat_vals):
    area_proportion_1 = yhat.sum() / len(yhat)
    area_proportion_0 = 1 - area_proportion_1
    
    # Figure out precision/false-omission-rate as the weighted-mean
    cms = [confusion_matrix(y_val, yhat_val, labels=[True, False]) for y_val, yhat_val in zip(y_vals, yhat_vals)] # (tp, fn), (fp, tn)
    tps = [e[0][0] for e in cms]
    fns = [e[0][1] for e in cms]
    tns = [e[1][1] for e in cms]
    fps = [e[1][0] for e in cms]

    # Figure out weighted-averaged TN, ...
    n_vals = np.array([len(e) for e in y_vals])
    tp = np.mean(tps)
    fn = np.mean(fns)
    fp = np.mean(fps)
    tn = np.mean(tns)

    # Compute validation set's metrics
    false_omission_rate = fn / (fn + tn)
    precision = tp / (tp + fp)

    # Compute adjusted unbiased area proportion
    ahat = (area_proportion_1 * precision) + (area_proportion_0 * false_omission_rate)

    # Compute Confidence Interval (Olofsson Variance)
    n_cocoa = tp + fp
    n_non_cocoa = fn + tn

    # Variance contribution from both strata
    var_cocoa = (area_proportion_1 ** 2) * (precision * (1 - precision)) / (n_cocoa - 1)
    var_non_cocoa = (area_proportion_0 ** 2) * (false_omission_rate * (1 - false_omission_rate)) / (n_non_cocoa - 1)
    
    # Standard Error
    se = np.sqrt(var_cocoa + var_non_cocoa)
    
    # Margin of Error for 95% Confidence Interval (Z-score = 1.96)
    margin_of_error = 1.96 * se
    
    ci_lower = max(0.0, ahat - margin_of_error) # Clamp at 0%
    ci_upper = min(1.0, ahat + margin_of_error) # Clamp at 100%

    return ahat, (ci_lower, ci_upper)

In [19]:
df = pd.read_parquet(f"../data/silver/ghana-{2021}-df.parquet")
df.region

1         Upper East
2         Upper East
3         Upper East
4         Upper East
5         Upper East
             ...    
963292       Western
963293       Western
963294       Western
963469       Western
963477       Western
Name: region, Length: 915540, dtype: category
Categories (16, str): ['Ahafo', 'Ashanti', 'Bono', 'Bono East', ..., 'Upper West', 'Volta', 'Western', 'Western North']

In [20]:
df = pd.read_parquet(f"../data/silver/ghana-{2021}-df.parquet")
df.region

1         Upper East
2         Upper East
3         Upper East
4         Upper East
5         Upper East
             ...    
963292       Western
963293       Western
963294       Western
963469       Western
963477       Western
Name: region, Length: 915540, dtype: category
Categories (16, str): ['Ahafo', 'Ashanti', 'Bono', 'Bono East', ..., 'Upper West', 'Volta', 'Western', 'Western North']

In [24]:
def predict(region: str, year: int):
    # Load model payload
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')

    global clf
    clf = payload['clf']
    user_attrs = payload['user_attrs']
    
    # Load input data
    df = pd.read_parquet(f"../data/silver/ghana-{year}-df.parquet")

    # Extract features
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl', "region"]
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    features += ['valid_observations']
    features += ['cloud_confidence']
    features += [e for e in df.columns if any(k in e for k in ['3x3', '5x5', '11x11'])]
    
    # Filter for the target region
    region_mask = df.region.apply(lambda x: x.lower()) == region.lower()
    X = df.loc[region_mask, features]

    # Run inference for EACH model in our ensemble
    # all_yhats will have shape: (n_models, n_samples)
    X['scl'] = pd.Categorical(X['scl'], categories=clf.pandas_categorical[0])
    X['region'] = pd.Categorical(X['region'], categories=clf.pandas_categorical[1])

    yhat = clf.predict(X)

    return yhat, user_attrs["y_vals"], user_attrs["yhat_vals"]

# Real area from Cocoa area estimation

My issue is that I'm dealing with 2 back-to-back area estimation.
Hence, for each region, I have

1. The real cocoa area
2. The ETHZ's cocoa area estimation
3. My cocoa area estimation

First, I need to figure out how to estimate the real cocoa area from the ETHZ's cocoa area estimation.

## Our estimate

The ETHZ's estimated area are the best we'll get.
We make the assumption that the ETHZ's estimate is correct, i.e

$$Â^{corrected}_{ETHZ} = A_{true}$$

From them, we'll compare our estimate, i.e.

$$A_{true} = Â^{corrected}_{ETHZ} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$
$$\rightarrow Â^{corrected}_{LGBM} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$

We'll measure how far off $Â^{corrected}_{ETHZ}$ we are

In [25]:
def compute_corrected_area_proportion(y, precision, recall):
    area_proportion = y.sum() / len(y)
    return area_proportion * precision / recall

In [26]:
def print_our_corrected_area_proportion(region: str, year: int) -> float:
    """Print the Corrected Area Proportion (CAP) of our prediction with a Bootstrapped Confidence Interval."""
    # Get the prediction matrix and metrics
    yhat, y_vals, yhat_vals = predict(region, year)

    ahat, (ci_lower, ci_upper) = compute_ahat(yhat, y_vals, yhat_vals)

    # Format the output bounds
    ap_str = f"{ahat*100:.2f}% [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]"

    print(f"{region.title()}'s entire region ({year}): Our cocoa area proportion (corrected): {ap_str}")
    
print_our_corrected_area_proportion('ashanti', 2022)
print_our_corrected_area_proportion('western north', 2022)
print_our_corrected_area_proportion('western', 2022)
print_our_corrected_area_proportion('ahafo', 2022)
print_our_corrected_area_proportion('central', 2022)

AttributeError: 'LGBMClassifier' object has no attribute 'pandas_categorical'

In [27]:
clf.pan

,num_leaves,78
,max_depth,13
,learning_rate,0.1726080942165134
,n_estimators,66
,min_child_samples,99
,subsample,0.7581595190307663
,colsample_bytree,0.9922859707323896
,boosting_type,'gbdt'
,subsample_for_bin,200000
,objective,None
,class_weight,None
